In [1]:
import importlib, sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G_0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

In [2]:
# fit values

GN_G0: float = 0.18877592218372993
Delta_meV: float = 0.19345000789195935
gamma_meV: float = 0.005066874981090785
eta: float = 0.002173
nu_GHz: float = 13.6

eta = 0.002173  # (3)
Aoff_mV = -6.2e-3  # (13) mV
Tbase_K = 0.0806  # (46) K
Toff_K = -0.685  # (194) K
alphaT = 2.6811  # (458)

In [3]:
# Location
from superconductivity.evaluation.keys import list_measurement_keys

importlib.reload(sys.modules["superconductivity.evaluation.keys"])

location = "/Users/oliver/Documents/measurement data/25 04 OI-25c-09"
h5path = (
    f"{location}/OI-25c-09 2025-05-02 unbroken stripline irradiation studies 0.hdf5"
)
measurement = "vna_amplitudes_18.3000GHz"

mkeys = list_measurement_keys(h5path=h5path)

In [4]:
# handle keys
from superconductivity.evaluation import list_specific_keys_and_values

importlib.reload(sys.modules["superconductivity.evaluation"])

keys, Aout_V = list_specific_keys_and_values(
    h5path=h5path,
    measurement=measurement,
    strip0="GHz_",
    strip1="V",
    remove_key="no_irradiation",
    add_key=[
        ("no_irradiation", 0.0),
        ("no_irradiation", 0.005),
    ],
    limits=(None, 0.1),
)
Aout_mV = Aout_V * 1e3

In [5]:
# get traces

from superconductivity.evaluation import get_ivs

importlib.reload(sys.modules["superconductivity.evaluation"])

traces = get_ivs(
    h5path=h5path,
    measurement=measurement,
    keys=keys,
    yvalues=Aout_mV,
    amp_voltage=1000,
    amp_current=1000,
    trigger_values=1,
    skip=5,
)

In [6]:
# psd analysis

from superconductivity.evaluation import get_psds

importlib.reload(sys.modules["superconductivity.evaluation"])

psdtraces = get_psds(traces=traces, detrend=True)

In [7]:
# offset analysis

from superconductivity.evaluation import get_offsets, OffsetSpec

importlib.reload(sys.modules["superconductivity.evaluation"])

offsetspec = OffsetSpec(
    Vbins_mV=np.linspace(-0.5, 0.5, 51),
    Ibins_nA=np.linspace(-5, 5, 181),
    Voff_mV=np.linspace(-0.045, 0.045, 451),
    Ioff_nA=np.linspace(-0.35, 0.35, 701),
    nu_Hz=13.7,
    upsample=10,
)
offsets = get_offsets(
    traces=traces,
    spec=offsetspec,
    backend="jax",
)

get_offsets:   0%|          | 0/15 [00:00<?, ?trace/s]

In [8]:
# sampling

from superconductivity.evaluation import get_samplings, SamplingSpec

importlib.reload(sys.modules["superconductivity.evaluation"])

spec = SamplingSpec(
    nu_Hz=13.7,
    upsample=1000,
    Vbin_mV=np.linspace(-1.6, 1.6, 1601),
    Ibin_nA=np.linspace(-30.0, 30.0, 2001),
)

samples = get_samplings(
    traces=traces,
    offsets=offsets,
    spec=spec,
)

Vbias_mV = samples.Vbin_mV
Ibias_nA = samples.Ibin_nA
Iexp_nA = samples.I_nA
Vexp_mV = samples.V_mV
dGexp_G0 = samples.dG_G0
dRexp_R0 = samples.dR_R0

get_samplings:   0%|          | 0/15 [00:00<?, ?trace/s]

In [9]:
# smoothing

from superconductivity.evaluation import get_smoothed_samplings, SmoothingSpec

importlib.reload(sys.modules["superconductivity.evaluation"])

smoothspec = SmoothingSpec(
    median_bins=5,
    sigma_bins=2.0,
)

smooth = get_smoothed_samplings(
    samplings=samples,
    spec=smoothspec,
)

Vbias_mV = smooth.Vbin_mV
Ibias_nA = smooth.Ibin_nA
Iexp_nA = smooth.I_nA
Vexp_mV = smooth.V_mV
dGexp_G0 = smooth.dG_G0
dRexp_R0 = smooth.dR_R0

In [ ]:
from superconductivity.optimizers.pat import fit_pat_gui

importlib.reload(sys.modules["superconductivity.optimizers.pat"])
solution = fit_pat_gui(
    V_mV=Vbias_mV,
    I_nA=Iexp_nA,
    model="conv_pat",
)

Panel server ready at http://127.0.0.1:63587 (Ctrl+C to stop and return the latest solution)
